# RiskIQ: Loan Risk Model Analysis
### Technical Deep Dive into the LightGBM Classifier

This notebook provides a detailed overview of the model used in the **RiskIQ Dashboard**. We will explore the architecture, feature importance, and the underlying logic of our Gradient Boosting Decision Tree (GBDT) implementation.

In [ ]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set paths
BASE_DIR = os.getcwd()
MODEL_PATH = os.path.join(BASE_DIR, 'models', 'perfect_gpu_model.pkl')

# Load Model Bundle
try:
    bundle = joblib.load(MODEL_PATH)
    model = bundle['model']
    encoders = bundle['encoders']
    features = bundle['features']
    params = bundle.get('params', {})
    print(f"✅ Model successfully loaded!")
    print(f"Features included: {len(features)}")
except Exception as e:
    print(f"❌ Failed to load model: {e}")

## 1. Model Architecture: LightGBM
LightGBM (Light Gradient Boosting Machine) is a framework developed by Microsoft. It is known for:
- **Leaf-wise tree growth**: Reduces more loss than level-wise growth.
- **GOSS (Gradient-based One-Side Sampling)**: Keeps instances with large gradients and performs random sampling on instances with small gradients.
- **EFB (Exclusive Feature Bundling)**: Reduces feature dimensionality by bundling mutually exclusive features.

In [ ]:
print("--- Model Hyperparameters ---")
for key, value in params.items():
    print(f"{key}: {value}")

print(f"\nBoosting Type: {params.get('boosting_type', 'gbdt')}")
print(f"Number of Trees (Iterations): {model.num_trees()}")

## 2. Feature Importance Analysis
Feature importance tells us which variables (e.g., Credit Score, Income) have the most predictive power in our model.

In [ ]:
importance = model.feature_importance(importance_type='gain')
fi_df = pd.DataFrame({'Feature': features, 'Importance': importance})
fi_df = fi_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=fi_df, palette='viridis')
plt.title('Feature Importance (Gain)')
plt.show()

## 3. Handling Categorical Variables
Our model uses `LabelEncoder` for features like `EmploymentType` and `LoanPurpose`. Here are the encoded classes:

In [ ]:
for feat, encoder in encoders.items():
    print(f"Feature: {feat}")
    print(f"Classes: {list(encoder.classes_)}\n")

## 4. Logical Interpretation
The model produces a **probability score** between 0 and 1.
- **Low Risk**: Predicted default probability < 0.5
- **High Risk**: Predicted default probability >= 0.5

The RiskIQ dashboard uses this score to calculate 'Confidence' by taking the distance from the 50% threshold.